# Brazilian Elections 2026 — Exploratory Data Analysis

Explore the local BRPOL Instagram posts, Meta Ad Library ads, and BRPOL aggregate package.
The notebook runs locally without API keys, network calls, model downloads, or LLM requests.
Source texts remain in Portuguese; notebook prose, code, tables, and chart labels use English.

**Questions:** What is the unit of each file? Which periods, accounts, states, source groups,
parties, media types, and language labels occur? How complete are texts and engagement metrics?
Which lexical patterns deserve inspection? Which texts could support controlled rewrite chains?

**Units must stay separate:** one post, one ad, one account-period, one account-day,
one aggregate stratum-day, one hashtag-period, and one TSE candidacy are different observations.
The aggregate tables summarize overlapping posts; do not concatenate them into the post corpus.
Source group labels describe accounts, not whether every individual post is election-related.

This is a collected corpus, not a random sample of Brazilian voters or all election news.
Engagement counts do not establish belief, sharing, factual veracity, or informational drift.
Political metadata are supplied labels, not inferred views of the author or reader.

Run cells from top to bottom in the project's Python environment. From the repository root:
`uv sync --locked`, then `uv run jupyter lab notebooks/elections_2026_eda.ipynb`.
Full post analyses read all rows but skip heavy multimedia payloads. Text models use a bounded,
seeded sample. All exports are local; review the embedded package README before redistribution.

## 1. Configuration and imports

Edit this cell to change chart limits, the text-analysis sample, the ad date window, or the
simulation screening rules. Equal quotas by source group and collection period create an
experimental review sample; they do not reproduce corpus proportions or population proportions.

In [ ]:
import importlib.metadata
import json
import platform
import re
import sys
import unicodedata
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import Markdown, display
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists() and (path / "src").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the notebook from the project root or its notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from misinformation_simulation.analysis.elections_2026 import (  # noqa: E402
    build_seed_pool,
    distribution,
    load_ads,
    load_aggregate_package,
    load_posts,
    parquet_inventory,
    parse_hashtags,
    parse_json_records,
    quality_profile,
    sample_seed_pool,
    sha256_file,
)

DATA_DIR = PROJECT_ROOT / "data" / "Eleições 2026"
OUTPUT_DIR = PROJECT_ROOT / "output" / "elections_2026"
SEED = 2026
TOP_N = 20
TEXT_SAMPLE_SIZE = 10_000
RUN_TOPIC_MODEL = True
TOPIC_SAMPLE_SIZE = 5_000
N_TOPICS = 8
AD_START = "2026-01-01"
AD_END = "2026-12-31"
SEED_GROUPS = ["veiculo_nacional", "veiculo_regional"]
SEED_LANGUAGES = ["pt"]
MIN_SEED_WORDS = 80
MAX_SEED_WORDS = 1_200
SAMPLE_PER_STRATUM = 10
MAX_PER_ACCOUNT = 2
EXPORT_ARTIFACTS = True

pd.set_option("display.max_columns", 35)
pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"
figures = {}
tables_to_export = {}


def show_figure(name, figure):
    figures[name] = figure
    figure.show()


def show_table(name, table, limit=30):
    tables_to_export[name] = table
    display(table.head(limit))


def show_distribution(column, frame=None):
    frame = posts if frame is None else frame
    table = distribution(frame, column)
    show_table(f"distribution_{column}", table)
    show_figure(
        f"distribution_{column}",
        px.bar(
            table.head(TOP_N),
            x=column,
            y="posts",
            hover_data=["share_pct", "active_accounts"],
            title=f"Collected posts by {column}",
            labels={"posts": "Post count"},
        ),
    )

## 2. Inventory, schemas, and source documentation

The ad CSV is nested under a directory whose name includes `23/09/2026`; recursive discovery
handles this layout. The ZIP is read in place, without extracting or changing the source files.
Check the embedded README for source provenance, collection limits, and use conditions.

In [ ]:
file_inventory = pd.DataFrame(
    [
        {"path": path.relative_to(DATA_DIR).as_posix(), "size_mb": path.stat().st_size / 1_000_000}
        for path in sorted(DATA_DIR.rglob("*"))
        if path.is_file()
    ]
)
show_table("file_inventory", file_inventory)
post_inventory, post_schema = parquet_inventory(DATA_DIR)
show_table("post_file_inventory", post_inventory)
show_table("post_schema", post_schema, limit=12)
schema_comparison = post_schema.pivot(index="column", columns="source_file", values="dtype")
display(schema_comparison.loc[schema_comparison.nunique(axis=1).gt(1)])

aggregate_tables, package_readme = load_aggregate_package(DATA_DIR)
display(Markdown(package_readme))
show_table(
    "aggregate_inventory",
    pd.DataFrame(
        [
            {"table": name, "rows": len(frame), "columns": len(frame.columns)}
            for name, frame in aggregate_tables.items()
        ]
    ),
)

## 3. Load posts and audit data quality

Numeric post metrics are stored as strings in the first Parquet and integers in later files.
Convert them explicitly; absent counts stay missing. Blank categorical fields become missing,
then appear as `Unknown` in distribution tables. IDs stay strings. The schema table covers all
original fields; only analytical columns are loaded into memory.

UTC is used for reconciliation with the source aggregates. Local dates use `America/Sao_Paulo`;
the first local day may be the preceding calendar date. Deduplication is audited before analysis.

In [ ]:
posts_loaded = load_posts(DATA_DIR)
post_quality = quality_profile(posts_loaded)
show_table("post_quality", post_quality)
duplicate_ids = posts_loaded.loc[
    posts_loaded["id"].notna() & posts_loaded["id"].duplicated(keep=False),
    ["id", "source_file", "creation_time", "baixado_em_utc"],
]
display(duplicate_ids.head(20))
# Keep the latest downloaded observation for repeated nonmissing IDs.
identified = posts_loaded.loc[posts_loaded["id"].notna()].sort_values("baixado_em_utc_utc")
posts = (
    pd.concat(
        [
            identified.drop_duplicates("id", keep="last"),
            posts_loaded.loc[posts_loaded["id"].isna()],
        ]
    )
    .sort_index()
    .reset_index(drop=True)
)
overview = pd.DataFrame(
    [
        {
            "loaded_rows": len(posts_loaded),
            "analysis_rows": len(posts),
            "duplicate_id_rows_removed": len(posts_loaded) - len(posts),
            "missing_ids": posts["id"].isna().sum(),
            "active_accounts": posts["conta_usuario"].nunique(),
            "first_publication_utc": posts["creation_time_utc"].min(),
            "last_publication_utc": posts["creation_time_utc"].max(),
            "empty_texts": posts["text"].isna().sum(),
            "repeated_nonempty_texts": posts.loc[posts["text_hash"].notna(), "text_hash"]
            .duplicated()
            .sum(),
            "invalid_publication_dates": posts["creation_time_utc"].isna().sum(),
            "memory_mb": posts.memory_usage(deep=True).sum() / 1_000_000,
        }
    ]
)
show_table("post_overview", overview)
metric_quality = pd.DataFrame(
    [
        {
            "metric": metric,
            "missing_rows": posts[metric].isna().sum(),
            "missing_pct": 100 * posts[metric].isna().mean(),
            "negative_rows": posts[metric].lt(0).sum(),
        }
        for metric in ["likes", "comments", "views", "idade_horas"]
    ]
)
show_table("metric_quality", metric_quality)

## 4. Publication volume and collection timing

Distinguish publication date, download date, and the date of the last views refresh. High
counts reflect this collection, not a census of all online activity. Collection periods have
different lengths; compare daily rates as well as totals. Do not infer trends from partial days
or infer audience response from raw posting volume.

In [ ]:
daily = (
    posts.groupby(["day_utc", "grupo"], observed=True)
    .agg(posts=("id", "size"), active_accounts=("conta_usuario", "nunique"))
    .reset_index()
)
show_table("posts_daily_utc", daily)
show_figure(
    "posts_daily_utc",
    px.line(
        daily,
        x="day_utc",
        y="posts",
        color="grupo",
        markers=True,
        title="Publication volume by source group (UTC)",
        labels={
            "day_utc": "Publication date (UTC)",
            "posts": "Post count",
            "grupo": "Source group",
        },
    ),
)
period_summary = (
    posts.groupby("collection_period", observed=True)
    .agg(
        posts=("id", "size"),
        observed_days=("day_utc", "nunique"),
        active_accounts=("conta_usuario", "nunique"),
        median_download_lag_hours=("download_lag_hours", "median"),
    )
    .reset_index()
)
period_summary["posts_per_observed_day"] = period_summary["posts"] / period_summary["observed_days"]
show_table("period_summary", period_summary)
local_daily = posts.groupby("day_local").size().reset_index(name="posts")
show_figure(
    "posts_daily_local",
    px.bar(
        local_daily,
        x="day_local",
        y="posts",
        title="Publication volume by local day (São Paulo)",
        labels={"day_local": "Local publication date", "posts": "Post count"},
    ),
)
refresh_dates = (
    posts.groupby(["collection_period", "statistics.views_date_last_refreshed"], observed=True)
    .size()
    .reset_index(name="posts")
)
show_table("views_refresh_dates", refresh_dates)

## 5. Sources, geography, and political metadata

Use both posts and active accounts: prolific accounts can dominate row counts. For party, office,
state, presidential field, and ideological labels, restrict to candidate accounts by default.
`campo_nacional`, `campo_disputa`, and `lado7` are distinct source metadata. News organizations
often have missing candidate metadata; those missing values are not political classifications.

In [ ]:
for column in ["grupo", "content_type", "lang", "tipo_coleta"]:
    show_distribution(column)
candidate_posts = posts.loc[posts["grupo"].eq("candidato")]
for column in ["uf", "cargo", "partido", "campo_nacional", "campo_disputa", "lado7", "genero"]:
    show_distribution(column, candidate_posts)
source_composition = pd.crosstab(posts["grupo"], posts["content_type"], normalize="index") * 100
show_table("source_media_share", source_composition.reset_index())
show_figure(
    "source_media_share",
    px.imshow(
        source_composition,
        text_auto=".1f",
        aspect="auto",
        title="Media type within each source group (%)",
        labels={"color": "Within-group share (%)"},
    ),
)

## 6. Account concentration

The cumulative curve shows how much of the corpus comes from the most prolific accounts.
This describes collection concentration, not influence. The top-account table includes a source
group label per account; inspect metadata conflicts separately before making account joins.

In [ ]:
account_volume = (
    posts.groupby("conta_usuario")
    .agg(
        posts=("id", "size"),
        source_groups=("grupo", "nunique"),
    )
    .sort_values("posts", ascending=False)
    .reset_index()
)
account_volume["account_rank"] = np.arange(1, len(account_volume) + 1)
account_volume["cumulative_post_share_pct"] = 100 * account_volume["posts"].cumsum() / len(posts)
show_table("account_volume", account_volume)
show_figure(
    "account_concentration",
    px.line(
        account_volume,
        x="account_rank",
        y="cumulative_post_share_pct",
        title="Cumulative post share ranked by account activity",
        labels={
            "account_rank": "Account rank",
            "cumulative_post_share_pct": "Cumulative post share (%)",
        },
    ),
)
show_table("account_metadata_conflicts", account_volume.loc[account_volume["source_groups"].gt(1)])
display(
    pd.DataFrame(
        [
            {
                "top_accounts": n,
                "post_share_pct": 100 * account_volume.head(n)["posts"].sum() / len(posts),
            }
            for n in [10, 100, 1000]
        ]
    )
)

## 7. Engagement coverage and distributions

Missing likes can be hidden, and stories do not provide the same metrics. Zero and missing are
different. Means and medians below use observed values, with the observed denominator displayed.
Views are a secondary signal and may refresh later. Metrics are snapshots, not standardized
exposure or engagement rates; follower denominators are not available.

Compare source groups within a collection period and inspect post age. Different downloads and
account sizes confound cross-period comparisons. `log1p` plots preserve zero while reducing the
visual dominance of outliers. Top-engagement rows are inspection examples, not a sampling rule.

In [ ]:
metric_long = posts[["collection_period", "grupo", "likes", "comments", "views"]].melt(
    id_vars=["collection_period", "grupo"], var_name="metric", value_name="value"
)
engagement_summary = (
    metric_long.groupby(["collection_period", "grupo", "metric"], observed=True)
    .agg(
        total_posts=("value", "size"),
        observed_posts=("value", "count"),
        mean_observed=("value", "mean"),
        median_observed=("value", "median"),
        p90_observed=("value", lambda values: values.quantile(0.9)),
    )
    .reset_index()
)
engagement_summary["coverage_pct"] = (
    100 * engagement_summary["observed_posts"] / engagement_summary["total_posts"]
)
show_table("engagement_summary", engagement_summary, limit=50)
plot_metrics = metric_long.sample(n=min(45_000, len(metric_long)), random_state=SEED)
plot_metrics = plot_metrics.loc[plot_metrics["value"].ge(0)].copy()
plot_metrics["log1p_count"] = np.log1p(plot_metrics["value"])
show_figure(
    "engagement_boxplots",
    px.box(
        plot_metrics,
        x="grupo",
        y="log1p_count",
        color="collection_period",
        facet_col="metric",
        points=False,
        title="Observed engagement counts (random row sample, log1p scale)",
        labels={"grupo": "Source group", "log1p_count": "log(1 + observed count)"},
    ),
)
lag_summary = (
    posts.groupby("collection_period", observed=True)["download_lag_hours"]
    .describe(percentiles=[0.1, 0.5, 0.9])
    .reset_index()
)
show_table("download_lag_summary", lag_summary)
display(
    posts.nlargest(10, "likes")[
        [
            "id",
            "conta_usuario",
            "grupo",
            "creation_time",
            "likes",
            "comments",
            "idade_horas",
            "text",
        ]
    ]
)
del metric_long, plot_metrics

## 8. Text availability, repeated captions, and hashtags

Only captions are available here; images, video speech, and linked article bodies are not
transcribed or fetched. A long caption can still lack the underlying factual claim. Language
labels come from the source and can be noisy. Text hashes normalize Unicode, case, and whitespace
for diagnostics and sampling, while preserving original text. Exact repetition after normalization
does not establish coordinated behavior, copying direction, or misinformation.

In [ ]:
text_summary = (
    posts.assign(has_text=posts["text"].notna())
    .groupby(["grupo", "content_type"], observed=True)
    .agg(posts=("id", "size"), texts=("has_text", "sum"), median_words=("word_count", "median"))
)
text_summary["text_coverage_pct"] = 100 * text_summary["texts"] / text_summary["posts"]
show_table("text_coverage", text_summary.reset_index())
text_plot = posts.sample(n=min(30_000, len(posts)), random_state=SEED)
show_figure(
    "text_length",
    px.histogram(
        text_plot,
        x="word_count",
        color="grupo",
        nbins=80,
        barmode="overlay",
        range_x=[0, 600],
        title="Caption word counts (random post sample; display limited to 600 words)",
        labels={"word_count": "Caption words"},
    ),
)
repeat_counts = posts["text_hash"].dropna().value_counts()
show_table(
    "text_repetition_summary",
    pd.DataFrame(
        [
            {
                "unique_nonempty_texts": len(repeat_counts),
                "text_hashes_repeated": repeat_counts.gt(1).sum(),
                "posts_in_repeated_text_groups": repeat_counts.loc[repeat_counts.gt(1)].sum(),
                "largest_repeat_group": repeat_counts.max(),
            }
        ]
    ),
)
display(
    posts.loc[
        posts["text_hash"].isin(repeat_counts.head(3).index),
        ["id", "conta_usuario", "grupo", "text"],
    ].head(15)
)
hashtag_counts = Counter()
for value in posts["hashtags"].dropna():
    hashtag_counts.update(parse_hashtags(value))
hashtags = pd.DataFrame(hashtag_counts.most_common(), columns=["hashtag", "posts"])
show_table("post_hashtags", hashtags)
show_figure(
    "post_hashtags",
    px.bar(
        hashtags.head(TOP_N),
        x="hashtag",
        y="posts",
        title="Top supplied hashtags (one count per post)",
    ),
)

## 9. Local lexical exploration and optional NMF topics

Use a fixed random sample of Portuguese-labeled captions with at least 20 words, after normalized
text deduplication. This sample differs from the full-row corpus; show its group composition.
The short, editable stopword list is a heuristic. URLs, mentions, digits, and accents are removed
only in the vectorizer; raw text stays unchanged. Terms measure document frequency, not truth.

NMF finds lexical co-occurrence patterns using TF-IDF. Its topic numbers are exploratory clusters,
not validated election themes, ideology labels, or veracity classes. A forced dominant assignment
hides multi-topic and weakly represented documents; inspect weights and original examples.
The fit covers only its sample; it is not a full-corpus theme distribution.

In [ ]:
STOPWORDS_PT = """
a ao aos aquela aquelas aquele aqueles aqui as ate com como contra da das de dela delas dele
deles depois do dos e ela elas ele eles em entre era eram essa essas esse esses esta estamos
estao estar estas este estes eu foi foram ha isso isto ja la lhe mais mas me mesmo meu meus
minha minhas muito na nas nao nem no nos nossa nossas nosso nossos num numa o os ou para pela
pelas pelo pelos por porque qual quando que quem se sem ser seu seus sua suas tambem tem tendo
tenho ter teve tu tudo um uma umas uns vai voce voces vou ser sera sao somos sou sobre ainda
agora hoje vamos pode podemos todos todas cada brasil brasileiro brasileira gente dia vez
""".split()


def prepare_lexical_text(value):
    value = re.sub(r"https?://\S+|www\.\S+|@\w+", " ", str(value))
    return "".join(
        character
        for character in unicodedata.normalize("NFKD", value.lower())
        if not unicodedata.combining(character)
    )


text_candidates = (
    posts.loc[posts["lang"].eq("pt") & posts["word_count"].ge(20) & posts["text_hash"].notna()]
    .drop_duplicates("text_hash")
    .sort_values("id")
)
text_sample = text_candidates.sample(
    n=min(TEXT_SAMPLE_SIZE, len(text_candidates)), random_state=SEED
)
display(
    pd.DataFrame(
        [{"eligible_unique_texts": len(text_candidates), "sampled_texts": len(text_sample)}]
    )
)
show_table("lexical_sample_composition", distribution(text_sample, "grupo"))
term_frequency = pd.DataFrame(columns=["term", "documents", "sample_share_pct"])
topic_summary = pd.DataFrame()
topic_assignments = pd.DataFrame()
vectorizer_options = dict(
    strip_accents="unicode",
    stop_words=STOPWORDS_PT,
    preprocessor=prepare_lexical_text,
    lowercase=True,
    token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ]{3,}\b",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85,
    max_features=12_000,
)
if len(text_sample) >= 10:
    count_vectorizer = CountVectorizer(binary=True, **vectorizer_options)
    try:
        document_terms = count_vectorizer.fit_transform(text_sample["text"])
    except ValueError as error:
        print(f"Lexical analysis skipped: {error}")
    else:
        term_frequency = pd.DataFrame(
            {
                "term": count_vectorizer.get_feature_names_out(),
                "documents": np.asarray(document_terms.sum(axis=0)).ravel(),
            }
        ).sort_values("documents", ascending=False)
        term_frequency["sample_share_pct"] = 100 * term_frequency["documents"] / len(text_sample)
        show_table("term_document_frequency", term_frequency)
        show_figure(
            "term_document_frequency",
            px.bar(
                term_frequency.head(TOP_N),
                x="term",
                y="sample_share_pct",
                title="Terms appearing in sampled captions",
                labels={"sample_share_pct": "Sample documents (%)"},
            ),
        )
else:
    print("Not enough eligible captions for lexical analysis")

if RUN_TOPIC_MODEL and len(text_sample) >= max(20, N_TOPICS):
    topic_sample = text_sample.sample(n=min(TOPIC_SAMPLE_SIZE, len(text_sample)), random_state=SEED)
    topic_vectorizer = TfidfVectorizer(**vectorizer_options)
    try:
        topic_matrix = topic_vectorizer.fit_transform(topic_sample["text"])
    except ValueError as error:
        print(f"Topic model skipped: {error}")
    else:
        topic_count = min(N_TOPICS, *topic_matrix.shape)
        model = NMF(n_components=topic_count, init="nndsvda", random_state=SEED, max_iter=400)
        weights = model.fit_transform(topic_matrix)
        terms = topic_vectorizer.get_feature_names_out()
        topic_summary = pd.DataFrame(
            [
                {
                    "topic": f"topic_{index}",
                    "top_terms": ", ".join(terms[component.argsort()[-12:][::-1]]),
                }
                for index, component in enumerate(model.components_)
            ]
        )
        weight_total = weights.sum(axis=1)
        topic_assignments = topic_sample[
            ["id", "grupo", "collection_period", "conta_usuario"]
        ].copy()
        topic_assignments["topic"] = [
            f"topic_{index}" if total > 0 else "Unrepresented"
            for index, total in zip(weights.argmax(axis=1), weight_total, strict=True)
        ]
        topic_assignments["dominant_weight_share"] = np.divide(
            weights.max(axis=1), weight_total, out=np.zeros(len(weights)), where=weight_total > 0
        )
        show_table("nmf_topic_terms", topic_summary)
        show_table("nmf_sample_assignments", topic_assignments)
        show_figure(
            "nmf_sample_topics",
            px.histogram(
                topic_assignments,
                x="topic",
                color="grupo",
                barmode="group",
                title=f"Dominant lexical topic in NMF sample (n={len(topic_sample):,})",
            ),
        )
        print(f"NMF iterations: {model.n_iter_}; configured maximum: {model.max_iter}")
        for topic in topic_summary["topic"]:
            selected = topic_assignments.loc[topic_assignments["topic"].eq(topic)].nlargest(
                2, "dominant_weight_share"
            )
            print(topic, topic_summary.loc[topic_summary["topic"].eq(topic), "top_terms"].iloc[0])
            display(topic_sample.loc[selected.index, ["id", "grupo", "text"]])

## 10. Meta Ad Library: dates, advertisers, platforms, and intervals

Ad delivery starts are dates of new ads, not daily impressions. Stop-date absence does not prove
that an ad remains active. The configurable window filters **delivery start**, not overlap or
exposure during the window. Inspect all years first: the filename is a collection date, not proof
that every ad began in 2026. Keep these observations separate from organic Instagram posts.

Spend and impressions are reported as ranges. Preserve lower and upper endpoints separately.
An absent upper bound is an open interval, not zero. Range endpoints are not statistical confidence
intervals. Sum spending only within a currency and disclose incomplete endpoints. Do not calculate
midpoints as exact spending or add ad impressions to organic views.

In [ ]:
ads_loaded = load_ads(DATA_DIR)
show_table("ad_quality", quality_profile(ads_loaded))
ad_id_duplicates = ads_loaded["ad_archive_id"].dropna().duplicated().sum()
ads = pd.concat(
    [
        ads_loaded.loc[ads_loaded["ad_archive_id"].notna()].drop_duplicates(
            "ad_archive_id", keep="last"
        ),
        ads_loaded.loc[ads_loaded["ad_archive_id"].isna()],
    ]
).reset_index(drop=True)
show_table(
    "ad_overview",
    pd.DataFrame(
        [
            {
                "loaded_ads": len(ads_loaded),
                "analysis_ads": len(ads),
                "repeated_id_rows": ad_id_duplicates,
                "advertiser_pages": ads["page_id"].nunique(),
                "first_delivery_start": ads["ad_delivery_start_time_utc"].min(),
                "last_delivery_start": ads["ad_delivery_start_time_utc"].max(),
                "missing_stop_dates": ads["ad_delivery_stop_time_utc"].isna().sum(),
            }
        ]
    ),
)
ad_years = (
    ads["ad_delivery_start_time_utc"].dt.year.value_counts(dropna=False).sort_index().reset_index()
)
ad_years.columns = ["delivery_start_year", "ads"]
show_table("ad_start_years", ad_years)
ads_window = ads.loc[
    ads["ad_delivery_start_time_utc"].between(
        pd.Timestamp(AD_START, tz="UTC"),
        pd.Timestamp(AD_END, tz="UTC") + pd.Timedelta(days=1),
        inclusive="left",
    )
].copy()
print(f"Ads whose delivery began between {AD_START} and {AD_END}: {len(ads_window):,}")
ad_monthly = (
    ads_window.assign(start_month=ads_window["ad_delivery_start_time_utc"].dt.strftime("%Y-%m"))
    .groupby("start_month")
    .size()
    .reset_index(name="ads")
)
show_table("ad_monthly_starts", ad_monthly)
show_figure(
    "ad_monthly_starts",
    px.bar(
        ad_monthly,
        x="start_month",
        y="ads",
        title="New ad delivery starts in the configured window",
    ),
)
ad_accounts = (
    ads_window.groupby(["page_id", "page_name"], dropna=False).size().reset_index(name="ads")
)
show_table("ad_advertisers", ad_accounts.sort_values("ads", ascending=False))
platform_counts = (
    ads_window["publisher_platforms"]
    .fillna("Unknown")
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
    .reset_index()
)
platform_counts.columns = ["platform", "ads"]
show_table("ad_platform_membership", platform_counts)
display(Markdown("Platform memberships overlap: an ad can appear on multiple platforms."))
ad_bounds_quality = pd.DataFrame(
    [
        {
            "metric": metric,
            "missing_lower": ads_window[f"{metric}_lower"].isna().sum(),
            "missing_upper": ads_window[f"{metric}_upper"].isna().sum(),
            "reversed_ranges": ads_window[f"{metric}_lower"]
            .gt(ads_window[f"{metric}_upper"])
            .sum(),
        }
        for metric in ["spend", "impressions", "estimated_audience_size"]
    ]
)
show_table("ad_bounds_quality", ad_bounds_quality)
ad_spend = (
    ads_window.groupby("currency", dropna=False)
    .agg(
        ads=("ad_archive_id", "size"),
        known_lower=("spend_lower", "count"),
        known_upper=("spend_upper", "count"),
        known_lower_sum=("spend_lower", lambda values: values.sum(min_count=1)),
        known_upper_sum=("spend_upper", lambda values: values.sum(min_count=1)),
    )
    .reset_index()
)
ad_spend["all_endpoints_known"] = ad_spend["known_lower"].eq(ad_spend["ads"]) & ad_spend[
    "known_upper"
].eq(ad_spend["ads"])
show_table("ad_spend_endpoints_by_currency", ad_spend)

## 11. Ad delivery metadata: regions and demographics

These CSV fields contain comma-separated JSON objects. Expand them as nested ad-level metadata,
not new ads. Percentages are per-ad platform estimates. Validate parse coverage and fraction sums.
The means below weight each reporting ad equally; they are **not** audience totals or actual
person-level demographic proportions. Ads with missing/invalid fields do not enter these means.
Only records with numerical fractions in [0, 1] enter the regional/demographic summaries.

In [ ]:
def expand_ad_field(column):
    rows, checks = [], []
    for ad_id, value in zip(ads_window["ad_archive_id"], ads_window[column], strict=True):
        records = parse_json_records(value)
        checks.append(
            {
                "ad_archive_id": ad_id,
                "field": column,
                "parsed_records": len(records),
                "source_nonmissing": pd.notna(value),
            }
        )
        rows.extend({**record, "ad_archive_id": ad_id} for record in records)
    expanded = pd.DataFrame(rows)
    if not expanded.empty:
        expanded["percentage"] = pd.to_numeric(expanded["percentage"], errors="coerce")
    return expanded, pd.DataFrame(checks)


regions, region_parse = expand_ad_field("delivery_by_region")
demographics, demographic_parse = expand_ad_field("demographic_distribution")
parse_checks = pd.concat([region_parse, demographic_parse], ignore_index=True)
parse_summary = (
    parse_checks.assign(
        parse_failure=lambda frame: frame["source_nonmissing"] & frame["parsed_records"].eq(0)
    )
    .groupby("field")
    .agg(
        ads=("ad_archive_id", "size"),
        fields_present=("source_nonmissing", "sum"),
        parse_failures=("parse_failure", "sum"),
        total_records=("parsed_records", "sum"),
    )
    .reset_index()
)
show_table("ad_nested_parse_summary", parse_summary)
for label, expanded, dimensions in [
    ("regions", regions, ["region"]),
    ("demographics", demographics, ["age", "gender"]),
]:
    if expanded.empty:
        print(f"No parsed {label} records in the configured ad window")
        continue
    sums = expanded.groupby("ad_archive_id")["percentage"].sum(min_count=1)
    show_table(
        f"ad_{label}_fraction_checks",
        pd.DataFrame(
            [
                {
                    "ads_with_records": len(sums),
                    "fraction_sum_outside_0_98_to_1_02": (~sums.between(0.98, 1.02)).sum(),
                    "invalid_fractions": (~expanded["percentage"].between(0, 1)).sum(),
                }
            ]
        ),
    )
    valid = expanded.loc[expanded["percentage"].between(0, 1)]
    summary = (
        valid.groupby(dimensions, dropna=False)
        .agg(
            reporting_ads=("ad_archive_id", "nunique"),
            mean_fraction_when_reported=("percentage", "mean"),
        )
        .reset_index()
        .sort_values("reporting_ads", ascending=False)
    )
    show_table(f"ad_{label}_metadata", summary)

## 12. Aggregate reconciliation and candidacy coverage

The ZIP covers only the first two post periods. Reconcile that overlap, not all three files.
Compare row counts with declared `posts` totals; never sum medians or assume every aggregate is
an independent observation. Hashtags in the ZIP have a >=20-post threshold and a shorter window.

TSE coverage uses the supplied `no_censo` flag. It describes inclusion in this collection, not
internet use, electoral support, or access to information. One row is a candidacy; do not join
the whole candidacy table onto posts without checking relationship cardinality.

In [ ]:
accounts_aggregate = aggregate_tables["contas"].copy()
accounts_aggregate["posts"] = pd.to_numeric(accounts_aggregate["posts"], errors="coerce")
declared_by_period = (
    accounts_aggregate.groupby("periodo")["posts"].sum(min_count=1).rename("aggregate_posts")
)
raw_by_period = posts.groupby("collection_period", observed=True).size().rename("raw_posts")
reconciliation = (
    pd.concat([raw_by_period, declared_by_period], axis=1).rename_axis("period").reset_index()
)
reconciliation["raw_minus_aggregate"] = (
    reconciliation["raw_posts"] - reconciliation["aggregate_posts"]
)
show_table("aggregate_reconciliation", reconciliation)
display(Markdown("Missing aggregate totals indicate a period outside the ZIP, not zero posts."))
tse = aggregate_tables["candidatos_tse"].copy()
tse["coverage_label"] = tse["no_censo"].fillna("Unknown").str.strip().str.lower()
show_table(
    "tse_coverage_overall", tse["coverage_label"].value_counts().reset_index(name="candidacies")
)
for column in ["uf", "cargo", "genero", "partido"]:
    coverage = (
        tse.groupby(column, dropna=False)
        .agg(
            candidacies=("sq_candidato", "size"),
            included=("coverage_label", lambda labels: labels.eq("sim").sum()),
            explicit_not_included=("coverage_label", lambda labels: labels.eq("nao").sum()),
            unknown=("coverage_label", lambda labels: (~labels.isin(["sim", "nao"])).sum()),
        )
        .reset_index()
    )
    coverage["included_share_pct"] = 100 * coverage["included"] / coverage["candidacies"]
    show_table(f"tse_coverage_{column}", coverage.sort_values("candidacies", ascending=False))
    if column == "uf":
        show_figure(
            "tse_coverage_uf",
            px.bar(
                coverage,
                x="uf",
                y="included_share_pct",
                hover_data=["candidacies", "included", "unknown"],
                title="TSE candidacies flagged as included, by state",
                labels={"included_share_pct": "Included candidacies (%)"},
            ),
        )

## 13. Reproducible candidates for future rewrite simulations

Default: national/regional news-account captions labeled Portuguese, 80–1,200 words, valid
publication/ID/account, excluding stories, unique post IDs and normalized text. Change
`SEED_GROUPS` to explore candidate or party discourse separately. A media source can publish
non-election content: **manually review election relevance, sufficient standalone context, and
the main claims before simulation**. Full article text may require a separate authorized source.

Sample up to 10 rows per source-group × collection-period stratum, with at most two rows per
account across the entire sample. A fixed seed reproduces this draw. Report shortfalls explicitly;
account caps and deduplication can prevent equal quotas. All rows initially have review status
`pending`. `category` is a source group, not a semantic topic or veracity label.

The CSV provides `description`, `article_id`, `source_post_id`, original text, source URL, dates,
and metadata. It is compatible with the existing interaction-graph CSV workflow; select
`description` as the text field. No simulation is run by this notebook. Use the **same approved
seed texts in every persona/order scenario** to enable paired comparisons. Store each model,
prompt, temperature, seed ID, persona order, and run date. STDI measures informational drift;
internal contradiction, factual veracity, belief, exposure, and sharing remain distinct.

In [ ]:
seed_pool, seed_filter_audit = build_seed_pool(
    posts,
    groups=SEED_GROUPS,
    languages=SEED_LANGUAGES,
    min_words=MIN_SEED_WORDS,
    max_words=MAX_SEED_WORDS,
)
simulation_sample, sampling_audit = sample_seed_pool(
    seed_pool,
    per_stratum=SAMPLE_PER_STRATUM,
    max_per_account=MAX_PER_ACCOUNT,
    seed=SEED,
)
show_table("simulation_filter_audit", seed_filter_audit)
show_table("simulation_sampling_audit", sampling_audit)
display(
    simulation_sample[
        [
            "article_id",
            "grupo",
            "collection_period",
            "conta_usuario",
            "word_count",
            "description",
            "manual_review_status",
        ]
    ].head(10)
)
assert simulation_sample["article_id"].is_unique
assert simulation_sample["text_hash"].is_unique
assert simulation_sample["conta_usuario"].value_counts().le(MAX_PER_ACCOUNT).all()
assert simulation_sample["word_count"].between(MIN_SEED_WORDS, MAX_SEED_WORDS).all()
assert sampling_audit["selected"].le(SAMPLE_PER_STRATUM).all()
print(
    f"Eligible unique texts: {len(seed_pool):,}; "
    f"sampled for manual review: {len(simulation_sample):,}"
)

## 14. Local exports and reproducibility manifest

Write tidy summaries, self-contained interactive HTML charts, a seed CSV for manual review, and
a manifest with source SHA-256 hashes, package versions, filters, quotas, and exclusions. The
source files are not edited. Hashes identify exact local inputs; they do not grant distribution
rights. Set `EXPORT_ARTIFACTS = False` for analysis without writing exports.

Outputs live in the ignored `output/elections_2026/` directory. Raw-text review candidates and
the executed notebook can contain restricted source content; apply the source README's stated
conditions before sharing. Rerunning overwrites files with the same names. Empty candidates are
reported rather than replaced with unsuitable texts.

In [ ]:
if EXPORT_ARTIFACTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, table in tables_to_export.items():
        table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, encoding="utf-8")
    for name, figure in figures.items():
        figure.write_html(OUTPUT_DIR / f"{name}.html", include_plotlyjs=True, full_html=True)
    simulation_sample.to_csv(
        OUTPUT_DIR / "simulation_candidates.csv", index=False, encoding="utf-8"
    )
    manifest = {
        "schema_version": 1,
        "created_at_utc": datetime.now(UTC).isoformat(),
        "python_version": platform.python_version(),
        "package_versions": {
            name: importlib.metadata.version(name)
            for name in ["pandas", "pyarrow", "plotly", "scikit-learn", "numpy"]
        },
        "source_files": [
            {
                "path": path.relative_to(PROJECT_ROOT).as_posix(),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
            for path in sorted(DATA_DIR.rglob("*"))
            if path.is_file()
        ],
        "loaded_posts": len(posts_loaded),
        "analyzed_posts": len(posts),
        "loaded_ads": len(ads_loaded),
        "ad_filter": {
            "field": "ad_delivery_start_time",
            "start": AD_START,
            "end_inclusive": AD_END,
        },
        "text_analysis": {
            "seed": SEED,
            "lexical_sample": len(text_sample),
            "topic_model_enabled": RUN_TOPIC_MODEL,
            "topic_sample_limit": TOPIC_SAMPLE_SIZE,
            "requested_topics": N_TOPICS,
            "stopwords": STOPWORDS_PT,
        },
        "simulation_sample": {
            "seed": SEED,
            "groups": SEED_GROUPS,
            "languages": SEED_LANGUAGES,
            "min_words": MIN_SEED_WORDS,
            "max_words": MAX_SEED_WORDS,
            "strata": ["grupo", "collection_period"],
            "per_stratum": SAMPLE_PER_STRATUM,
            "max_per_account": MAX_PER_ACCOUNT,
            "eligible_rows": len(seed_pool),
            "selected_rows": len(simulation_sample),
            "selected_article_ids": simulation_sample["article_id"].tolist(),
            "review_status": "pending",
            "normalized_text_deduplication": "NFKC, casefold, whitespace",
        },
        "artifacts": sorted(
            [f"{name}.csv" for name in tables_to_export]
            + [f"{name}.html" for name in figures]
            + ["simulation_candidates.csv", "manifest.json"]
        ),
    }
    (OUTPUT_DIR / "manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(
        f"Wrote {len(tables_to_export)} tables, {len(figures)} charts, "
        f"candidates, and manifest to {OUTPUT_DIR}"
    )
else:
    print("Exports disabled")

## 15. Interpretation checklist and next experiments

- State the unit, denominator, collection period, and missingness for each finding.
- Separate candidate discourse, news-account captions, and paid ads when defining experiments.
- Inspect lexical topic examples and refine a manually reviewed theme codebook if needed.
- Keep cohort selection independent of engagement when testing persona effects.
- Freeze reviewed seed IDs and source hashes; reuse the same seeds across conditions.
- For a pilot, choose a few sufficient-context texts, then inspect rewrite quality and drift components.
- Treat platform counts and collection coverage as descriptive; causal effects and factual judgments
  require a separate design and evidence.

This notebook does not modify the thesis source, automatically synchronize Overleaf, retrieve
linked content, or label posts as true/false. Its local results are findings of this dataset analysis.